# BER 1/2: full pipeline (CPU)

This notebook runs everything from the raw challenge data up to the cross-encoder package:
prep → blocking → round-1 models → round-2 expansion → test-like retrain (simulate) → package for the GPU notebook.

**Setup (once)**
- **Add Input** (right panel) → **Datasets**: add your private dataset **`ber-inputs`**. It must contain the 7 challenge `.tsv` files (with or without the `train/`, `test/` folders) and `ber_code.zip`.
- **Settings**: Accelerator = **None** (CPU; saves GPU hours), **Internet = On** (for pip).
- Then click **Save Version → Save & Run All (Commit)**. This runs in the background for about 6–7 hours, and you can close the laptop.
- When it has finished, its **Output** (`ce_pkg/` plus `out_sim/`) is what notebook 2/2 uses.

In [ ]:
import glob, os, subprocess, sys, time
t_src = glob.glob('/kaggle/input/**/train_source1.tsv', recursive=True)
z_src = glob.glob('/kaggle/input/**/ber_code.zip', recursive=True) + glob.glob('/kaggle/input/**/ber_code/src/run_pipeline.py', recursive=True)
print(t_src, z_src)
assert t_src and z_src, 'add the ber-inputs dataset (dataset/ folder + ber_code.zip) as input'
if os.path.basename(os.path.dirname(t_src[0])) == 'train':
    DATA = os.path.dirname(os.path.dirname(t_src[0]))
else:   # files were uploaded without the train/ test/ folders -> link them into that layout
    DATA = '/tmp/ber_data'
    for split in ('train', 'test'):
        os.makedirs(f'{DATA}/{split}', exist_ok=True)
        for f in glob.glob(f'/kaggle/input/**/{split}_*.tsv', recursive=True):
            dst = f'{DATA}/{split}/{os.path.basename(f)}'
            if not os.path.exists(dst): os.symlink(f, dst)
print(sorted(glob.glob(f'{DATA}/*/*.tsv')))
CODE = '/kaggle/working/code'
if z_src[0].endswith('.zip'):
    subprocess.run(['unzip', '-o', '-q', z_src[0], '-d', CODE], check=True)
else:
    subprocess.run(['cp', '-r', os.path.dirname(os.path.dirname(z_src[0])), CODE], check=True)
if not os.path.exists(f'{CODE}/src'):   # zip contained a top-level folder
    inner = [d for d in glob.glob(f'{CODE}/*') if os.path.exists(f'{d}/src')][0]; CODE = inner
TMP = '/kaggle/temp'
try:
    os.makedirs(TMP, exist_ok=True); open(f'{TMP}/.w', 'w').close()
except OSError:
    TMP = '/tmp'
WORK, SIM = f'{TMP}/work', f'{TMP}/sim'   # large scratch (not saved); /kaggle/working keeps only outputs
print('DATA', DATA, '| CODE', CODE, '| cpus', os.cpu_count())

In [ ]:
!pip install -q -r {CODE}/requirements.txt 2>&1 | grep -v -i warning | tail -3

In [ ]:
def run(*args):
    t = time.time(); print('>>', ' '.join(args), flush=True)
    p = subprocess.run([sys.executable, *args], cwd=CODE)
    print(f'<< exit {p.returncode} after {(time.time()-t)/60:.1f} min', flush=True)
    assert p.returncode == 0

run('src/run_pipeline.py', '--data', DATA, '--work', WORK, '--out', '/kaggle/working/out_v2')

In [ ]:
run('src/simulate.py', '--data', DATA, '--work', WORK, '--sim', SIM, '--out', '/kaggle/working/out_sim')

In [ ]:
run('src/ce_prep.py', '--work', WORK, '--sim', SIM, '--out', '/kaggle/working/ce_pkg')
!cp {CODE}/src/ce_train.py /kaggle/working/ce_pkg/
!ls -la /kaggle/working/ce_pkg /kaggle/working/out_sim
!cat {SIM}/validation_report.json